# Causal ML for AI Evaluation: calibrating an LLM judge to expert labels


ODSC AI West 2026, hands-on lab · Eddie Landesberg (Applied AI, Google; views are my own) · I started CJE, an open-source library, before joining Google · CJE is joint work with Manjari Narayan ([arXiv 2512.11150](https://arxiv.org/abs/2512.11150))

Open in Colab: **[https://colab.research.google.com/github/cimo-labs/cje/blob/main/examples/workshops/odsc-2026/workshop.ipynb](https://colab.research.google.com/github/cimo-labs/cje/blob/main/examples/workshops/odsc-2026/workshop.ipynb)** (folder: https://github.com/cimo-labs/cje/tree/main/examples/workshops/odsc-2026) · `cje-eval==0.9.3` · Python 3.10-3.13 (not 3.14) · no API keys, no model calls

**Run the first code cell first.** In Colab or a fresh environment it installs `cje-eval` and `matplotlib` if they are missing, and downloads the `lab/` data folder if it is not next to this notebook. After that the notebook runs offline in seconds.

**By the end you can:**
1. Calibrate LLM-judge scores to a small random sample of expert labels, and estimate each system's mean on the label scale with an interval.
2. Read a paired comparison (sign, units, interval) and decide against 0, with the rule written down before running anything.
3. Tell a support problem (`REFUSE-LEVEL`) from a transport problem (audit `FAIL` or `INCONCLUSIVE`), and name the next step for each.
4. Spend a fixed label budget two ways (independent audit vs correction) and say what each buys.
5. Size a transport audit with `plan_transport_audits`, and run the same workflow on your own CSV.

**Workflow:** Calibrate → Estimate → Audit → Correct.

In [ ]:
# Copyright (c) 2026 CJE contributors · SPDX-License-Identifier: MIT
# Setup: run this first. Installs what's missing and fetches lab/ if this notebook was opened on its own.
import os, sys, json, shutil, subprocess, warnings, importlib, urllib.request, urllib.error
from importlib.metadata import version, PackageNotFoundError
from importlib.util import find_spec
from pathlib import Path

# Public copy of this folder: examples/workshops/odsc-2026 in the cje repo (the folder behind the Colab link).
LAB_BASE = os.environ.get("CJE_LAB_BASE", "https://raw.githubusercontent.com/cimo-labs/cje/main/examples/workshops/odsc-2026/")

if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    warnings.warn(f"Python {sys.version.split()[0]}: cje-eval==0.9.3 requires Python 3.10-3.13 "
                  "(pinned install stops with 'No matching distribution' on 3.9 and 3.14; a bare "
                  "'pip install cje-eval' on 3.9 quietly installs legacy 0.5). Create a 3.12 env "
                  "(python3.12 -m venv .venv, or uv venv -p 3.12) or open the Colab link above.")

missing = [pkg for pkg, module in (("cje-eval==0.9.3", "cje"), ("matplotlib", "matplotlib")) if find_spec(module) is None]
if missing:
    print("Installing", " ".join(missing), "...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
    importlib.invalidate_caches()

if not Path("lab/workshop-data.json").exists():
    if LAB_BASE.startswith("http"):
        Path("lab").mkdir(exist_ok=True)
        def fetch(name):
            url = LAB_BASE.rstrip("/") + "/lab/" + name
            try:
                with urllib.request.urlopen(url, timeout=60) as response:
                    (Path("lab") / name).write_bytes(response.read())
            except Exception as exc:
                if not any(Path("lab").iterdir()):
                    shutil.rmtree("lab", ignore_errors=True)
                raise RuntimeError(
                    f"Could not download {url} ({exc}). Clone the repo or unzip the lab folder "
                    "and open workshop.ipynb from inside examples/workshops/odsc-2026/."
                ) from exc
        fetch("artifact-hashes.json")  # the file list; the next cell checks every hash
        for name in json.loads(Path("lab/artifact-hashes.json").read_text()):
            fetch(name)
        print("Downloaded lab/ from", LAB_BASE)
    else:
        warnings.warn("lab/ is not next to this notebook. Unzip the lab folder (link on the first slide) "
                      "and open workshop.ipynb from inside it.")

try:
    CJE_VERSION = version("cje-eval")
except PackageNotFoundError:
    CJE_VERSION = None
print(f"Python {sys.version.split()[0]} · cje-eval {CJE_VERSION} · "
      f"matplotlib {'installed' if find_spec('matplotlib') else 'missing (figures are skipped; numbers still print)'}")
if CJE_VERSION is None:
    warnings.warn("cje-eval did not install. Use Python 3.10-3.13 (`pip install -r lab/requirements.txt`) or open the Colab link.")
elif CJE_VERSION != "0.9.3":
    warnings.warn(f"This lab is pinned to cje-eval 0.9.3; you have {CJE_VERSION}, so outputs may differ.")

## 1. Write the decision before the code

- **Question:** switch from summarizer **M10** (current) to **M11** (candidate)?
- **Estimand:** each system's mean expert *consistency* rating (1-5, mean of 3 experts) on this document population.
- **Contrast:** M11 − M10, in rating points.
- **Decision rule:** switch to M11 only if CJE marks the result decision-ready *and* the paired 95% lower bound clears 0 plus any audit tolerance the comparison relies on; keep M10 if the whole interval is below 0; otherwise, no supported switch. (A `PASS` on a system that borrows a calibration still allows bias up to its tolerance. The extra is 0 when every system has its own random labels.)
- **Audit tolerance, ±0.10 points:** a reused calibration passes transport only if its mean error is shown to sit inside ±0.10.
- **Data:** public SummEval expert ratings and G-Eval's cached GPT-4 judge scores for summaries of CNN/DailyMail news articles. Chosen because every response has a human label, so we can check the answer at the end. Ratings pile up at 5.
- **Words:** document = one news article; both systems summarize it. (The slides say article.)
- **Design, on purpose:** this week's candidate scored with last week's calibration. 30 random M10 labels, none on M11. (The README asks for 20+ random labels per policy; we break that rule to see what CJE does.) 40 more documents are held out and labeled on both systems, for the audit only.

In [ ]:
import json, hashlib, math
from pathlib import Path
import numpy as np

AUDIT_TOLERANCE = 0.10  # points: allowed mean error of a reused calibration
VERBOSE = False         # True prints the full summaries and metadata behind each table

LAB = Path("lab")
DATA_SHA256 = "01d66d572c9e5e235534f0885ba5c0ea0847fea7c8843cd672397e61fe4ec85c"  # the published extract
if not (LAB / "workshop-data.json").exists():
    raise FileNotFoundError("lab/workshop-data.json not found: run the setup cell, or open the notebook from the lab folder.")
data_bytes = (LAB / "workshop-data.json").read_bytes()
assert hashlib.sha256(data_bytes).hexdigest() == DATA_SHA256, "lab/workshop-data.json is not the published extract"
listed = json.loads((LAB / "artifact-hashes.json").read_text())
changed = [n for n, h in listed.items() if (LAB / n).exists() and hashlib.sha256((LAB / n).read_bytes()).hexdigest() != h]
assert not changed, f"changed since publication: {changed}"
absent = [n for n in listed if not (LAB / n).exists()]
if absent:
    print("Listed in lab/artifact-hashes.json but not here (the core lab only needs workshop-data.json):", absent)

payload = json.loads(data_bytes)
draws, probes = payload["draws"], payload["probes"]  # evaluation rows; held-out audit rows

def ids(rows):
    return {r["prompt_id"] for r in rows}

# Leakage guards: both systems answer the same documents; audit documents never touch the fit.
assert ids(draws["M10"]) == ids(draws["M11"]) and ids(probes["M10"]) == ids(probes["M11"])
assert ids(draws["M10"]).isdisjoint(ids(probes["M10"]))

def table(header, rows):
    widths = [max(len(str(x)) for x in col) for col in zip(header, *rows)]
    for line in [header, ["-" * w for w in widths], *rows]:
        print("  ".join(str(x).ljust(w) for x, w in zip(line, widths)).rstrip())

def to_points(x):  # the bundle stores (score - 1) / 4; convert back to the native 1-5 scale
    return None if x is None else 1 + 4 * x

def n_labels(rows):
    return sum(r.get("oracle_label") is not None for r in rows)

def raw_judge_mean(rows):
    return 1 + 4 * sum(r["judge_score"] for r in rows) / len(rows)

table(["", "M10", "M11"], [
    ["evaluation documents", len(draws["M10"]), len(draws["M11"])],
    ["  with expert labels (fit)", n_labels(draws["M10"]), n_labels(draws["M11"])],
    ["audit documents (held out)", len(probes["M10"]), len(probes["M11"])],
    ["  with expert labels (audit)", n_labels(probes["M10"]), n_labels(probes["M11"])],
    ["raw judge mean, eval docs (1-5; descriptive only)", f"{raw_judge_mean(draws['M10']):.3f}", f"{raw_judge_mean(draws['M11']):.3f}"],
])

# Helpers used in sections 3-6 to format CJE rows, audit states, and our predeclared decision rule:
def cje_rows(rows, parser="midpoint"):
    key = {"midpoint": "judge_score", "lower": "score_lower", "upper": "score_upper"}[parser]
    return [{"prompt_id": r["prompt_id"], "judge_score": to_points(r[key]),
             "oracle_label": to_points(r.get("oracle_label"))} for r in rows]

def borrows(res, policy):
    """True if the policy's estimate rests on another policy's calibration (no own-label correction)."""
    routes = dict(zip(res.metadata["target_policies"], res.metadata["point_estimator"]["routes"]))
    return routes.get(policy) not in ("augmented", "direct_oracle")

def relied_tolerance(res, a, b):
    """Summed tolerance of the PASS audits this comparison relies on (0 if both have their own labels)."""
    audits = res.metadata.get("transport_audits") or {}
    return sum(audits[s]["delta_max"] for s in (a, b)
               if borrows(res, s) and audits.get(s, {}).get("status") == "PASS")

def action_from(lower, upper, ready=True, tol=0.0):
    """Our predeclared rule: switch only on a decision-ready win whose lower bound clears 0 + relied-on tolerance."""
    if ready and lower > tol:
        return "switch to M11"
    if ready and upper < 0:
        return "keep M10"
    return "no supported switch: get more evidence"

def decide(res):
    c = res.compare_policies("M11", "M10")  # difference = M11 - M10, in points
    verdict = res.best_policy()             # verdict.decision_ready is CJE's tested gate
    tol = relied_tolerance(res, "M11", "M10")
    return c, verdict, action_from(c["ci_lower"], c["ci_upper"], verdict.decision_ready, tol)

def ci_text(est, lower, upper):
    return f"{est:+.3f} [{lower:+.3f}, {upper:+.3f}]"

def states(res):
    cards = res.metadata["boundary_cards"]
    audits = res.metadata.get("transport_audits") or {}
    support = " / ".join(cards[s]["status"] for s in ("M10", "M11"))
    graded = all(audits.get(s, {}).get("graded") for s in ("M10", "M11"))
    transport = " / ".join(audits[s]["status"] for s in ("M10", "M11")) if graded else "not checked"
    passable = " / ".join(f"±{audits[s]['min_margin_for_pass']:.2f}" for s in ("M10", "M11")) if graded else "-"
    return support, transport, passable

def audit_line(res, s):
    a = res.metadata["transport_audits"][s]
    lo_, hi_ = a["delta_ci"]
    role = "borrows M10's map" if borrows(res, s) else "own labels, so this gates nothing"
    return f"{a['delta_hat']:+.2f} [{lo_:+.2f}, {hi_:+.2f}] vs ±{a['delta_max']:.2f}: {a['status']} ({role})"

## 2. Look before you fit (Checkpoint 1)

- Each judge score averages 20 GPT-4 samples; count articles, not samples.
- **Left:** M10's 30 labeled responses and the calibration fit on them: isotonic regression, the best non-decreasing curve from judge score to expert rating.
- **Right:** where M11's judge scores fall. The shaded zones had no labels, so the map there is a flat guess (dashed): every M11 score below 1.55 gets 3.0.
- Before running the next section: what will CJE say about M11?

In [ ]:
from sklearn.isotonic import IsotonicRegression  # installed with cje-eval

fit_rows = [r for r in draws["M10"] if r["oracle_label"] is not None]
s_fit = np.array([to_points(r["judge_score"]) for r in fit_rows])
y_fit = np.array([to_points(r["oracle_label"]) for r in fit_rows])
s_m11 = np.array([to_points(r["judge_score"]) for r in draws["M11"]])
lo, hi = s_fit.min(), s_fit.max()
outside = (s_m11 < lo) | (s_m11 > hi)
iso = IsotonicRegression(out_of_bounds="clip").fit(s_fit, y_fit)
print(f"Labeled judge-score range: {lo:.2f} to {hi:.2f}. "
      f"M11 scores outside it: {outside.sum()} of {len(s_m11)} ({outside.mean():.1%}).")

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed, so the figures are skipped. Every number still prints.")

INK, MUTED, AXIS, GRID, SHADE = "#0b0b0b", "#52514e", "#c3c2b7", "#e1e0d9", "#ecebe6"
BLUE, ORANGE = "#2a78d6", "#eb6834"
if plt is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3), facecolor="white")
    for ax in (ax1, ax2):
        ax.axvspan(0.9, lo, color=SHADE, lw=0, zorder=0)
        ax.axvspan(hi, 5.1, color=SHADE, lw=0, zorder=0)
        ax.set_xlim(0.9, 5.1)
        ax.set_xticks([1, 2, 3, 4, 5])
        ax.grid(axis="y", color=GRID, lw=0.8)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["left", "bottom"]].set_color(AXIS)
        ax.tick_params(colors=MUTED, labelsize=10)
        ax.set_xlabel("GPT-4 judge score (1-5, mean of 20 samples)", color=MUTED)

    grid = np.linspace(0.9, 5.1, 841)
    inside_grid = (grid >= lo) & (grid <= hi)
    ax1.scatter(s_fit, y_fit, s=46, color=BLUE, alpha=0.8, edgecolor="white", linewidth=1.2, zorder=3,
                label="M10 response with an expert label")
    ax1.plot(grid[inside_grid], iso.predict(grid[inside_grid]), color=INK, lw=2, zorder=4,
             label="isotonic calibration")
    ax1.plot(grid[grid < lo], iso.predict(grid[grid < lo]), color=INK, lw=2, ls=(0, (3, 2)), zorder=4)
    ax1.plot(grid[grid > hi], iso.predict(grid[grid > hi]), color=INK, lw=2, ls=(0, (3, 2)), zorder=4)
    ax1.set_ylim(0.8, 5.25)
    ax1.set_yticks([1, 2, 3, 4, 5])
    ax1.set_ylabel("expert consistency (1-5, mean of 3)", color=MUTED)
    ax1.set_title("M10: 30 expert labels and the calibration fit on them", loc="left", color=INK, fontsize=12)
    ax1.legend(frameon=False, loc="lower right", fontsize=9, labelcolor=MUTED)

    edges = 1.525 + 0.2 * np.arange(-3, 19)  # scores step by 0.05, so edges fall between attainable values
    ax2.hist(s_m11[~outside], bins=edges, color=ORANGE, edgecolor="white", lw=1.5, zorder=3,
             label="M11, inside the labeled range")
    ax2.hist(s_m11[outside], bins=edges, facecolor="#f7c3ad", edgecolor=ORANGE, hatch="///", lw=1.0, zorder=3,
             label="M11, outside the labeled range")
    ax2.set_ylabel("M11 evaluation documents", color=MUTED)
    ax2.yaxis.get_major_locator().set_params(integer=True)
    ax2.set_title(f"M11: {outside.mean():.1%} of judge scores fall outside that range", loc="left",
                  color=INK, fontsize=12)
    ax2.legend(frameon=False, loc="upper right", fontsize=9, labelcolor=MUTED)
    for ax in (ax1, ax2):
        ax.text(lo, 1.0, f" labels start at {lo:.2f}", transform=ax.get_xaxis_transform(), va="top",
                ha="left", fontsize=8.5, color=MUTED)
    fig.tight_layout()
    Path("figures").mkdir(exist_ok=True)
    fig.savefig("figures/calibration.png", dpi=200, bbox_inches="tight")
    plt.show()

## 3. Two checks before reading any number

- **Support:** do the candidate's judge scores sit inside the range the labels covered? If 5% or more fall outside, CJE marks the policy `REFUSE-LEVEL`: no level claims from this fit. (`CAUTION` means 20% or more of calibrated values sit at the ends of the calibration's fitted range; here mostly at 5, some at the map's 3.0 floor.)
- **Transport:** does last week's map still have zero mean error on this week's system? Held-out labels grade it against the **audit tolerance, ±0.10**:
  - `PASS`: the interval for the map's mean error (expert − map) sits wholly inside ±0.10, with at least 20 independent prompt clusters.
  - `FAIL`: it sits wholly outside.
  - `INCONCLUSIVE`: it crosses the tolerance (or the audit is too small to pass). Not a pass.
- The audit matters for a system that **borrows** the map. A system corrected by its own random labels does not rely on transport, so its audit gates nothing.
- Support is not transport: scores can sit in range while the map is still wrong for the new system.

> **Before you run the next cell:** the summary ends with `Status: critical`. That means validity checks failed. Your install is fine. The pink boxes are CJE's own warnings, printed on purpose.
>
> CJE's words: oracle label = expert label · policy = system · probe = audit article.

In [ ]:
from cje import analyze_dataset, TransportAuditConfig

SCALES = dict(fresh_judge_scale=(1, 5), fresh_oracle_scale=(1, 5))  # native units: declare your rubric's bounds

def run(budget=30, parser="midpoint"):
    """The lab design: M10 labels from a fixed random order, M11 borrows M10's map, 40-document audit."""
    keep = set(payload["label_order"][:budget])  # fixed random label order; never reshuffled
    data = {s: cje_rows(rows, parser) for s, rows in draws.items()}
    for r in data["M10"]:
        if r["prompt_id"] not in keep:
            r["oracle_label"] = None
    audit = TransportAuditConfig(
        probes_by_policy={s: cje_rows(rows, parser) for s, rows in probes.items()},
        delta_max_by_policy={"M10": AUDIT_TOLERANCE, "M11": AUDIT_TOLERANCE})
    return analyze_dataset(fresh_draws_data=data, transport=audit, **SCALES)

result = run()
print(result.summary())
c, verdict, action = decide(result)
support, transport, _ = states(result)
print("\n(Sign note: summary() above prints M10 - M11 = +0.556; our decision record below uses M11 - M10 = -0.556.)\n")
table(["decision record", ""], [
    ["design", "30 M10 labels, M11 borrowed, 40-document audit"],
    ["labels spent", "30 fit + 80 audit = 110"],
    ["M11 - M10, points [95% CI]", ci_text(c["difference"], c["ci_lower"], c["ci_upper"])],
    ["support (M10 / M11)", support],
    ["M11 audit: expert - map", audit_line(result, "M11")],
    ["M10 audit: expert - map", audit_line(result, "M10")],
    ["decision-ready (CJE)", "yes" if verdict.decision_ready else "no"],
    ["action (our rule)", action],
])
print("Audit intervals are simultaneous 95% across the two audits (97.5% each).")
print("\nCJE's note:", verdict.decision_note)

# The figure's isotonic curve is CJE's fitted map (checked on 0.9.3).
if "iso" in globals() and not np.allclose(iso.predict(np.linspace(1, 5, 401)),
                                          result.calibrator.predict(np.linspace(1, 5, 401))):
    print("Note: the figure's curve differs from CJE's fitted map in this version.")
if VERBOSE:
    for key in ("point_estimator", "boundary_cards", "transport_audits", "pairwise_inference"):
        print(f"\n{key}:", json.dumps(result.metadata.get(key), indent=1, default=str))

## 4. Read the result

- **Sign:** the summary prints `M10 - M11: +0.556`. Our contrast, M11 − M10, is −0.556 points, 95% CI [−1.583, +0.471].
- **What the interval is:** the paired t-test you know, plus uncertainty from fitting the calibration on 30 labels. That fit uncertainty is why df ≈ 7.
- **M10** is corrected by its own labels, so its estimate does not rely on transport. Its `INCONCLUSIVE` audit only says 40 documents can't pass ±0.10; it gates nothing.
- **M11** rests entirely on M10's map: `[borrowed calibration]`. Its audit is the one that matters, and it says `FAIL`.
- `Limitations: residual transport INCONCLUSIVE` in the summary is M10's audit (the point-estimate leader); M11's is the `FAIL` above.
- Intervals aren't clipped to the 1-5 scale, so M11's level interval reaches 5.011.
- **Plain words:** the interval assumes M10's calibration transfers to M11. The audit says it does not, and 16.7% of M11's scores sit outside the labeled range (`REFUSE-LEVEL`).
- **Decision:** no supported switch. Not evidence that M10 and M11 are equal.
- Next cell: the paired half-width. At about ±1 point, this design could not have said "switch" for any realistic gain.

In [ ]:
half_width = (c["ci_upper"] - c["ci_lower"]) / 2
print(f"Paired half-width: {half_width:.2f} points (df {c['df']:.1f}).")

## Exercise A′: the same 110 expert labels, used differently (hands-on)

We hold 110 expert labels: 30 M10 fit labels plus 80 audit labels (40 documents × 2 systems). The lab design spent 80 of them on an independent audit. Now spend **all 110 as correction labels**: CJE fits and corrects with M10's 70 and M11's 40, across all 100 documents. CJE's `FAIL` advice is to label random responses of M11, re-run without the failed probes in `transport=`, and keep the failed audit on record. Its audit-correction guide adds that a `FAIL` does not stop random labels on M11 from correcting M11's estimate, so here M11's 40 audit labels become its correction labels. They stop counting as validation.

1. **Predict** before running: does the interval narrow? Is it decision-ready? What do we give up?
2. **Run** the next cell.
3. **Explain** row 3: the 80 audit labels alone, compared directly with a paired t-test. No judge.

The audit documents and the label order were fixed random draws, so each system's labeled documents are still a random subset of the 100. Once M11 has its own random labels, its estimate no longer relies on transport.

In [ ]:
from scipy import stats  # installed with cje-eval

# Row 2: every one of the 110 labels becomes a correction label, on all 100 documents.
spend = {s: cje_rows(draws[s]) + cje_rows(probes[s]) for s in ("M10", "M11")}
spent = analyze_dataset(fresh_draws_data=spend, **SCALES)  # no transport=: no independent labels remain
c2, verdict2, action2 = decide(spent)

# Row 3: no judge at all. Paired t-test on the 40 audit documents (80 labels).
m11_label = {r["prompt_id"]: to_points(r["oracle_label"]) for r in probes["M11"]}
gaps = np.array([m11_label[r["prompt_id"]] - to_points(r["oracle_label"]) for r in probes["M10"]])
se = gaps.std(ddof=1) / math.sqrt(len(gaps))
t_crit = stats.t.ppf(0.975, len(gaps) - 1)
direct = (gaps.mean(), gaps.mean() - t_crit * se, gaps.mean() + t_crit * se)

if VERBOSE:
    print(spent.summary(), "\n")
designs = [
    ("Lab design", "30 + 0 (+80 audit)", "60 eval", (c["difference"], c["ci_lower"], c["ci_upper"]),
     transport, "yes" if verdict.decision_ready else "no", action),
    ("Exercise A′", "70 + 40", "all 100", (c2["difference"], c2["ci_lower"], c2["ci_upper"]),
     "none left", "yes" if verdict2.decision_ready else "no", action2),
    ("Direct, no judge", "40 + 40 (80)", "40 audit", direct,
     "none left", "n/a (no judge)", action_from(direct[1], direct[2])),
]
table(["design", "labels M10 + M11", "documents", "M11 - M10 [95% CI]", "audit (M10 / M11)", "ready", "action"],
      [[d, n, docs, ci_text(*est), a, r, act] for d, n, docs, est, a, r, act in designs])
print("Each row estimates the gap on a different document set, so the rows need not agree.")

if plt is not None:
    fig, ax = plt.subplots(figsize=(8, 2.8), facecolor="white")
    for i, (name, _, _, (est, lower, upper), *_rest) in enumerate(designs):
        y = len(designs) - 1 - i
        ax.plot([lower, upper], [y, y], color=BLUE, lw=2, solid_capstyle="round")
        ax.scatter([est], [y], s=60, color=BLUE, edgecolor="white", linewidth=1.5, zorder=3)
        ax.text(upper + 0.05, y, f"{est:+.2f} [{lower:+.2f}, {upper:+.2f}]".replace("-", "−"), va="center", fontsize=9, color=MUTED,
                bbox=dict(facecolor="white", edgecolor="none", pad=1), zorder=5)
    ax.axvline(0, color=AXIS, lw=1)
    labels = [f"{d}\n{n} labels; {docs} docs" for d, n, docs, *_ in designs]
    ax.set_yticks(range(len(designs)), labels[::-1], color=INK, fontsize=9)
    ax.set_ylim(-0.6, len(designs) - 0.3)
    ax.set_xlim(-2.2, 1.3)
    ax.set_xlabel("M11 − M10, expert consistency points (95% CI)", color=MUTED)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color(AXIS)
    ax.tick_params(axis="x", colors=MUTED)
    ax.tick_params(axis="y", length=0)
    ax.set_title("Same label pool, three designs", loc="left", color=INK, fontsize=12)
    fig.tight_layout()
    fig.savefig("figures/three-designs.png", dpi=200, bbox_inches="tight")
    plt.show()

**What to notice**
- Same labels, different job. Spent on correction, they make the answer decision-ready: keep M10.
- Each system now has its own random labels, so this estimate needs no transport assumption. What we gave up is evidence about whether M10's map can be reused on an unlabeled system next week. Labels used for correction are no longer independent validation, and the M11 `FAIL` stays on the record.
- Row 3 matches row 2 with 30 fewer labels because it is fully paired on the same 40 documents. And here the judge has little to extend to: 70 of 100 M10 documents and 40 of 100 M11 documents carry labels, and fitting the map costs some width. A judge pays off when labels are a small slice of a large pool.

## 5. Let the planner size the audit

How many held-out expert labels would it take to check M10's map on M11 well enough to `PASS` at ±0.10? `plan_transport_audits` answers from the spread the audit just measured, assuming the map's true mean error is 0. That is the best case, and for M11 it is not true. M10's audit gates nothing (its own labels correct it), so the plan covers M11 alone.

- A `PASS` also needs at least 20 independent prompt clusters. Below that the audit reads `INCONCLUSIVE`, though a clear `FAIL` is still graded.
- Acting on a `PASS` at tolerance δ: CJE's `decision_ready` already requires the lower bound to clear δ (summed over the systems that borrow).

In [ ]:
from cje import plan_transport_audits, AuditScenario, CostModel

audits = result.metadata["transport_audits"]
resid_sd = {s: audits[s]["delta_se"] * math.sqrt(audits[s]["n_clusters"]) for s in ("M10", "M11")}
print("Spread of expert - map per audit article (points):", {s: round(v, 2) for s, v in resid_sd.items()}, "\n")

plan_rows = []
for tol in (0.10, 0.20, 0.40):
    plan = plan_transport_audits(
        {"M11": AuditScenario(residual_sd=resid_sd["M11"], delta_max=tol)},  # M11 alone: M10's audit gates nothing
        calibration_labels=30,
        cost_model=CostModel(surrogate_cost=0.01, oracle_cost=1.0),  # cost unit: one label
    )
    plan_rows.append([f"±{tol:.2f}", f"{plan.policies['M11'].required_labels:,.0f}"])
table(["audit tolerance", "expert labels to check M10's map on M11"], plan_rows)
print("\n80% modeled power to PASS, best case (zero true bias). Today's audit used 40 articles per system. "
      "Exercise A′ decided with the same 110 labels: M10 70, M11 40.")
if VERBOSE:
    print("\n" + plan.summary())

## 6. Your data (Monday)

Four fields per response: `prompt_id`, the policy (here `variant`), `judge_score`, and a label (blank = unlabeled). The next cell is the README's CSV recipe with the scales declared, run on `lab/toy-evals.csv`: 60 prompts × 2 policies, 30 random labels per policy, 1-5 scale. **Constructed numbers, not real data** (`lab/make-toy-evals.py` made them). Swap in your own export.

- **Labels:** a random sample of each policy's responses, 20+ per policy, never hand-picked. To cover the score tails, sample at the *same rate* in every judge-score bin. If you oversample some range, record each row's inclusion probability and pass `label_design="known_propensity"`; otherwise the estimate and its interval can be biased, with no warning. Thumbs-up feedback and escalations are selected, not sampled.
- **Weekly releases:** label 20-30 random responses of each new candidate, spread that way across its judge-score range.
- **Ship margin:** on your own data, write down the smallest gain worth switching for before you run anything (`TOY_SHIP_MARGIN` below), and ship only on a decision-ready win whose lower bound clears it.
- **Declare your rubric's bounds** (`fresh_judge_scale`, `fresh_oracle_scale`). Without them, adding `transport=` can stop on a scale error.
- Log the judge model, rubric version and prompt set with every score. Re-audit when the judge, rubric, system, prompt mix or outcome changes.

**From Promptfoo** (TruLens and OpenCompass work the same way). The converters live in the GitHub repo, not the pip package:

```sh
git clone --branch v0.9.3 https://github.com/cimo-labs/cje.git && cd cje
python3 scripts/cje_bridges/convert.py promptfoo ../results.json --out ../draws.json --label-template ../labels.csv
# pick rows with the sampler below, have experts fill oracle_label in ../to_label.csv, then:
python3 scripts/cje_bridges/convert.py promptfoo ../results.json --oracle-labels ../to_label.csv \
    --out ../draws_with_labels.json --no-label-template
```

Pick which rows to label (pandas; same rate in every judge-score bin, seeded):

```python
import pandas as pd
t = pd.read_csv("labels.csv")  # the converter's template: one row per response
t["bin"] = t.groupby("provider_id")["judge_score"].transform(lambda s: pd.qcut(s.rank(method="first"), 5, labels=False))
t.groupby(["provider_id", "bin"]).sample(n=5, random_state=2026).drop(columns="bin").to_csv("to_label.csv", index=False)
```

Then: `analyze_dataset(fresh_draws_data=json.load(open("draws_with_labels.json")), fresh_judge_scale=(0, 1), fresh_oracle_scale=(lo, hi))`. Promptfoo scores are 0-1; use your label scale for `(lo, hi)`. Policies are named by Promptfoo provider ID.

**LangSmith:** the converter reads labels only from LangSmith feedback. Log your labels there as feedback and pass `--oracle-feedback-key <key>`; it does not read a labels CSV back. **Langfuse:** `cje.bridges.langfuse` ships in the pip package; see the Langfuse guide linked below.

In [ ]:
import csv
from collections import defaultdict
from cje import analyze_dataset

TOY_SCALE = (1, 5)        # this file's judge and label scale: declare your rubric's bounds
TOY_SHIP_MARGIN = 0.10    # in this file's label units (1-5 points); set yours on your label's scale

csv_draws = defaultdict(list)  # (the README calls this `draws`; renamed so the lab data above survives)
with open("lab/toy-evals.csv") as f:  # prompt_id, variant, judge_score, human_rating (blank = unlabeled)
    for row in csv.DictReader(f):
        csv_draws[row["variant"]].append({
            "prompt_id": row["prompt_id"],
            "judge_score": float(row["judge_score"]),
            "oracle_label": float(row["human_rating"]) if (row["human_rating"] or "").strip() else None,
        })
results = analyze_dataset(fresh_draws_data=dict(csv_draws), fresh_judge_scale=TOY_SCALE, fresh_oracle_scale=TOY_SCALE)
# Reusing a calibration on a policy with no labels? Add transport=TransportAuditConfig(probes_by_policy=...,
# delta_max_by_policy=...) with held-out random labels on that policy, as in section 3.
print(results.summary())

# A CI gate: ship only on a decision-ready win that clears your margin plus any audit tolerance relied on.
v, c_toy = results.best_policy(), results.compare_policies("candidate", "production")
tol = relied_tolerance(results, "candidate", "production")  # 0 here: both policies have their own labels
ship = v.name == "candidate" and v.decision_ready and c_toy["ci_lower"] > TOY_SHIP_MARGIN + tol
record = {"cje_version": CJE_VERSION, "ship": ship, "result": results.to_dict()}
# Path("cje-record.json").write_text(json.dumps(record, default=str))
print("\nship:", ship, "|", v.decision_note)

- With labels on fewer than 4 prompts, CJE falls back to the raw judge mean, marks it `UNCALIBRATED`, and says so.
- Bridges for Promptfoo, TruLens, LangSmith and OpenCompass: [scripts/cje_bridges](https://github.com/cimo-labs/cje/blob/v0.9.3/scripts/cje_bridges/README.md). Langfuse: [scripts/langfuse_cje](https://github.com/cimo-labs/cje/blob/v0.9.3/scripts/langfuse_cje/README.md).
- Many variants: paired p-values are unadjusted. Predeclare the contrast, or adjust (for example, Holm).
- Label budget before you collect: [planning notebook](https://colab.research.google.com/github/cimo-labs/cje/blob/v0.9.3/examples/cje_planning.ipynb) (no data needed), `plan_for_mde`, `plan_evaluation`.

**Coding agent.** `cje skill` prints the procedure and six hard rules: never report a raw judge mean; random labels only; one fixed judge; decide from the paired test; no winner and no lean on a borrowed calibration; surface every gate and audit state. It cannot check that your labels were random or that the judge stayed fixed: you write that line. The cje README's paste-in block, pinned to 0.9.3:

```text
Install cje-eval in a Python 3.10-3.13 environment: pip install -U "cje-eval>=0.9.3" pandas
(on Python 3.9 a bare `pip install cje-eval` silently installs the legacy 0.5 line).
Run `cje skill` and follow it; when it points to reference.md, run `cje skill --reference`.
If `cje` is unavailable, curl the raw files from
https://raw.githubusercontent.com/cimo-labs/cje/v0.9.3/skills/cje/ (SKILL.md, reference.md)
rather than a summarizing web fetcher.
Data: [path to the eval export]
Question: [e.g. is candidate better than production on support prompts, and by how much?]
How the labels were chosen: [e.g. 50 random responses per policy rated by our QA team /
escalated tickets only / not sure]
```

## Extras (if you finish early)

- **E1. Fewer labels (30 → 20).** Predict first. Watch M10's support: fewer labels cover a narrower score range.
- **E2. Parser sensitivity.** A few judge replies did not parse cleanly; the bundle carries a low and a high reading for those. Does the decision change?
- **E3. Exercise B: which bias changes the choice?** Pencil first, in the E3 cells further down (Exercise B, in the deck's appendix).

In [ ]:
import logging
logging.getLogger("cje").setLevel(logging.WARNING if VERBOSE else logging.ERROR)  # the table, without repeat warnings
small = run(budget=20)
low, high = run(parser="lower"), run(parser="upper")
extra_rows = []
for name, res in [("lab (30 labels, midpoint)", result), ("E1: 20 labels", small),
                  ("E2: parser low", low), ("E2: parser high", high)]:
    ce, ve, ae = decide(res)
    sup, tra, _ = states(res)
    extra_rows.append([name, ci_text(ce["difference"], ce["ci_lower"], ce["ci_upper"]), sup, tra,
                       "yes" if ve.decision_ready else "no", ae])
    if VERBOSE:
        print(f"--- {name}\n{res.summary()}\n")
logging.getLogger("cje").setLevel(logging.NOTSET)
table(["run", "M11 - M10 [95% CI]", "support (M10 / M11)", "transport (M10 / M11)", "ready", "action"], extra_rows)

### E3. Exercise B: which bias changes the choice? (pencil first)

Same constructed systems as the opening slides, on a 0-1 scale. The slides showed raw judge means (0.80 / 0.86); these are after calibration (0.78 / 0.82), where the judge says B beats A by 0.04. A residual is a system's mean error: expert minus calibrated.

Fill in the three blanks, then run.

In [ ]:
pred_A, pred_B = 0.78, 0.82  # calibrated judge means
r_A, r_B = -0.02, -0.12      # mean residual (expert - calibrated) on each system's own responses

expert_A = ...               # fill in: the expert mean for A
expert_B = ...               # fill in: the expert mean for B
common = ...                 # fill in: expert B - A if both residuals were -0.10 instead

def show(x):
    return "(fill in)" if x is ... else f"{x:+.2f}"

print("Calibrated judge, B - A:", show(pred_B - pred_A))
print("Experts, B - A:         ", "(fill in)" if ... in (expert_A, expert_B) else show(expert_B - expert_A))
print("Common residual, B - A: ", show(common))

## Answer key (facilitator)

Off by default. The full SummEval/G-Eval panel is public but is **not** shipped with this lab. To turn the key on, download [`results/gpt4_con_detailed.json` at G-Eval commit 6f84404](https://raw.githubusercontent.com/nlpyang/geval/6f84404ce4503d2ba0cde0293c5d402ecf2976a8/results/gpt4_con_detailed.json) and set `PANEL_PATH` below (or the `CJE_WORKSHOP_PANEL` environment variable). The cell checks the file's hash and that it reproduces the bundle's 110 labels. With the key on it also saves `figures/three-designs-answer.png`, the slide version of the figure with each row's true gap.

In [ ]:
import os
PANEL_PATH = os.environ.get("CJE_WORKSHOP_PANEL", "")  # facilitator: path to gpt4_con_detailed.json
PANEL_SHA256 = "e09609ab249fc6aa2e4f11b7c16d7531b7036b163c8251e54a04ce4c3efa76ed"  # G-Eval @6f84404

if not PANEL_PATH:
    print("Answer key is off. The facilitator shows it on the slide.")
else:
    raw = Path(PANEL_PATH).read_bytes()
    assert hashlib.sha256(raw).hexdigest() == PANEL_SHA256, "not the pinned G-Eval file"
    truth = {(r["doc_id"], r["system_id"]): r["scores"]["consistency"] for r in json.loads(raw)}
    bundle_labels = [(r["prompt_id"], s, to_points(r["oracle_label"]))
                     for part in (draws, probes) for s in ("M10", "M11") for r in part[s]
                     if r["oracle_label"] is not None]
    mismatches = sum(abs(truth[(d, s)] - y) > 1e-9 for d, s, y in bundle_labels)
    print(f"Panel verified: {len(truth):,} responses; the bundle's {len(bundle_labels)} labels match "
          f"({mismatches} mismatches).\n")

    eval_docs, audit_docs = sorted(ids(draws["M10"])), sorted(ids(probes["M10"]))
    doc_sets = {"60 evaluation": eval_docs, "40 audit": audit_docs, "all 100": eval_docs + audit_docs}

    def mean(docs, system):
        return float(np.mean([truth[(d, system)] for d in docs]))

    def true_gap(docs):
        d = np.array([truth[(x, "M11")] - truth[(x, "M10")] for x in docs])
        half = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / math.sqrt(len(d))
        return d.mean(), d.mean() - half, d.mean() + half

    table(["documents", "M10", "M11", "true M11 - M10 [paired t 95% CI]"],
          [[k, f"{mean(v, 'M10'):.2f}", f"{mean(v, 'M11'):.2f}", ci_text(*true_gap(v))]
           for k, v in doc_sets.items()])
    raw_m10, raw_m11 = raw_judge_mean(draws["M10"]), raw_judge_mean(draws["M11"])
    print(f"\nRaw judge, 60 evaluation docs: M10 {raw_m10:.3f}, M11 {raw_m11:.3f}, gap {raw_m11 - raw_m10:+.3f}. "
          "Right direction; levels about a point low.\n")

    def inside(ci, value):
        return "inside" if ci[1] <= value <= ci[2] else "outside"

    gap60, gap100 = true_gap(eval_docs)[0], true_gap(doc_sets["all 100"])[0]
    rows_ = [
        ["Lab design", ci_text(*designs[0][3]), f"{gap60:+.2f} (60 docs, {inside(designs[0][3], gap60)})"],
        ["Exercise A′", ci_text(*designs[1][3]), f"{gap100:+.2f} (100 docs, {inside(designs[1][3], gap100)})"],
        ["Direct, no judge", ci_text(*designs[2][3]), "exact on its 40 docs by construction"],
    ]
    table(["design", "estimate [95% CI]", "truth on the documents it estimates"], rows_)
    print(f"Row 3 against all 100 docs ({gap100:+.2f}): {inside(designs[2][3], gap100)}.")
    print(f"Every design points the same way (M11 is worse). Each row estimates its own documents: the 40 audit docs "
          f"show a wider gap ({true_gap(audit_docs)[0]:+.2f}) than the other 60 ({gap60:+.2f}), so rows 2 and 3 sit lower.\n")

    est = dict(zip(result.metadata["target_policies"], result.estimates))
    plug = dict(zip(result.metadata["target_policies"], result.metadata["point_estimator"]["plug_in_estimates"]))
    err10, err11 = est["M10"] - mean(eval_docs, "M10"), est["M11"] - mean(eval_docs, "M11")
    print(f"Lab design levels vs truth (60 docs): M10 {est['M10']:.2f} vs {mean(eval_docs, 'M10'):.2f} ({err10:+.2f}; "
          f"its map alone read {plug['M10']:.2f}), M11 {est['M11']:.2f} vs {mean(eval_docs, 'M11'):.2f} ({err11:+.2f}). "
          f"The same 30-label map read high on both, so the contrast was off by only {err11 - err10:+.2f}: "
          "a mostly common error cancels.")
    print(f"Exercise B: expert A {pred_A + r_A:.2f}, expert B {pred_B + r_B:.2f}, B - A {(pred_B + r_B) - (pred_A + r_A):+.2f} "
          f"(the winner flips); common residual: {pred_B - pred_A:+.2f} (no flip).")

    if plt is not None:  # slide version of the three-designs figure, with each row's true gap
        truths = [gap60, gap100, true_gap(audit_docs)[0]]
        fig, ax = plt.subplots(figsize=(8.6, 3.4), facecolor="white")
        minus = lambda x: f"{x:+.2f}".replace("-", "\u2212")
        for i, ((name, n, docs, (e, lower, upper), *_rest), t) in enumerate(zip(designs, truths)):
            y = len(designs) - 1 - i
            ax.plot([lower, upper], [y, y], color=BLUE, lw=2, solid_capstyle="round")
            ax.scatter([t], [y], s=150, facecolor="white", edgecolor=INK, linewidth=1.6, zorder=3)
            ax.scatter([e], [y], s=50, color=BLUE, edgecolor="white", linewidth=1.2, zorder=4)
            ax.text(upper + 0.05, y, f"{minus(e)} [{minus(lower)}, {minus(upper)}]   truth {minus(t)}",
                    va="center", fontsize=9, color=MUTED,
                    bbox=dict(facecolor="white", edgecolor="none", pad=1), zorder=5)
        ax.scatter([], [], s=50, color=BLUE, label="estimate and 95% CI")
        ax.scatter([], [], s=150, facecolor="white", edgecolor=INK, linewidth=1.6,
                   label="true gap on that row's documents")
        ax.axvline(0, color=AXIS, lw=1)
        labels = [f"{d}\n{n} labels; {docs} docs" for d, n, docs, *_ in designs]
        ax.set_yticks(range(len(designs)), labels[::-1], color=INK, fontsize=9)
        ax.set_ylim(-0.6, len(designs) - 0.3)
        ax.set_xlim(-2.2, 1.9)
        ax.set_xlabel("M11 − M10, expert consistency points (95% CI)", color=MUTED)
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.spines["bottom"].set_color(AXIS)
        ax.tick_params(axis="x", colors=MUTED)
        ax.tick_params(axis="y", length=0)
        ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(0.0, -0.3), ncol=2, fontsize=8.5, labelcolor=MUTED)
        ax.set_title("Same label pool, three designs: each row has its own documents and its own truth",
                     loc="left", color=INK, fontsize=11)
        fig.tight_layout()
        fig.savefig("figures/three-designs-answer.png", dpi=200, bbox_inches="tight")
        plt.show()

## Debrief

- Write the decision before the code: estimand, contrast, decision rule.
- Read the paired difference, not two overlapping intervals.
- Support and transport are different checks. `INCONCLUSIVE` is not a pass. A system with its own random labels doesn't need transport. "No supported switch" is not equivalence.
- Labels spent on correction are no longer independent validation.
- Size the audit before buying labels.
- Calibrated expert ratings are not business value. Linking them to a KPI is its own study (for example, across past A/B tests). CJE runs ahead of an A/B test, not instead of one.

## Sources and notices

- CJE: [github.com/cimo-labs/cje](https://github.com/cimo-labs/cje) (MIT). Landesberg and Narayan, *Causal Judge Evaluation: Calibrated Surrogate Metrics for LLM Systems*, [arXiv 2512.11150](https://arxiv.org/abs/2512.11150).
- G-Eval, source of the cached GPT-4 judge scores and the ratings file: [nlpyang/geval @6f84404](https://github.com/nlpyang/geval/tree/6f84404ce4503d2ba0cde0293c5d402ecf2976a8). MIT License, Copyright (c) 2024 Yang Liu. Full notice: `lab/LICENSE-upstream.txt`.
- SummEval, source of the expert consistency ratings: [Yale-LILY/SummEval](https://github.com/Yale-LILY/SummEval). MIT License, Copyright (c) 2021 Alex Fabbri, Wojciech Kryściński, Bryan McCann, Caiming Xiong, Richard Socher, and Dragomir Radev. Full notice: `lab/LICENSE-summeval.txt`.
- `lab/workshop-data.json` holds only document and system IDs, numeric judge scores (GPT-4 outputs released by the G-Eval authors) and aggregate expert ratings. No article or summary text.
- Surrogate index: Athey, Chetty, Imbens and Kang (2019), NBER Working Paper 26463.
- Prediction-powered inference: Angelopoulos, Bates, Fannjiang, Jordan and Zrnic (2023), *Science*. Relatives: PPI++ (Angelopoulos, Duchi and Zrnic, arXiv 2311.01453), cross-prediction-powered inference (Zrnic and Candès, arXiv 2309.16598), AutoEval Done Right (Boyeau et al., arXiv 2403.07008), and predictions as surrogates (Ji, Lei and Zrnic, arXiv 2501.09731).